In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [2]:
#!/bin/bash
!kaggle datasets download ashishjangra27/face-mask-12k-images-dataset

Dataset URL: https://www.kaggle.com/datasets/ashishjangra27/face-mask-12k-images-dataset
License(s): CC0-1.0
100% 330M/330M [00:02<00:00, 132MB/s]



In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
import random
import cv2
from keras.layers import Dense, Flatten, BatchNormalization
from keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import zipfile
from keras.layers import GlobalAveragePooling2D
# from keras.applications.vgg16 import VGG16
# from keras.applications.resnet50 import ResNet50
from keras.applications.efficientnet import EfficientNetB0
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping

In [4]:
# load data

In [5]:

zip_file = zipfile.ZipFile('/content/face-mask-12k-images-dataset.zip')
zip_file.extractall('/content')
zip_file.close()

In [25]:
# Create augmentation generator
train_datagen = ImageDataGenerator(
    # rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
)

test_datagen = ImageDataGenerator()
validation_datagen = ImageDataGenerator()

In [26]:
train_data = train_datagen.flow_from_directory(
    '/content/Face Mask Dataset/Train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary')

test_data = test_datagen.flow_from_directory(
    '/content/Face Mask Dataset/Test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary')

validation_data = validation_datagen.flow_from_directory(
    '/content/Face Mask Dataset/Validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary')

Found 10000 images belonging to 2 classes.
Found 992 images belonging to 2 classes.
Found 800 images belonging to 2 classes.


In [22]:
eff = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [23]:
eff.summary()

Model: "efficientnetb0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer_3[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,049,571 (15.45 MB)

 Trainable params: 4,007,548 (15.29 MB)

 Non-trainable params: 42,023 (164.16 KB)

In [27]:
model = Sequential()
model.add(eff)
model.add(GlobalAveragePooling2D())
model.add(Dense(64, activation='relu'))
model.add(BatchNormalization())
model.add(Dense(1, activation='sigmoid'))

In [28]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,131,876 (15.76 MB)

 Trainable params: 4,089,725 (15.60 MB)

 Non-trainable params: 42,151 (164.66 KB)

In [30]:
model.compile(optimizer=Adam(learning_rate=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

In [31]:
call = EarlyStopping(monitor='val_loss', patience=2, verbose=1)

In [33]:
model.fit(train_data, validation_data=validation_data, epochs=4, callbacks=call, batch_size=64)

Epoch 1/4
313/313 ━━━━━━━━━━━━━━━━━━━━ 250s 663ms/step - accuracy: 0.8739 - loss: 0.2940 - val_accuracy: 0.9538 - val_loss: 0.1909
Epoch 2/4
313/313 ━━━━━━━━━━━━━━━━━━━━ 144s 459ms/step - accuracy: 0.9746 - loss: 0.0911 - val_accuracy: 0.9800 - val_loss: 0.0871
Epoch 3/4
313/313 ━━━━━━━━━━━━━━━━━━━━ 145s 462ms/step - accuracy: 0.9836 - loss: 0.0606 - val_accuracy: 0.9887 - val_loss: 0.0590
Epoch 4/4
313/313 ━━━━━━━━━━━━━━━━━━━━ 146s 468ms/step - accuracy: 0.9874 - loss: 0.0449 - val_accuracy: 0.9887 - val_loss: 0.0450


In [ ]:
model.evaluate(test_data)

In [35]:
train_data.class_indices

{'WithMask': 0, 'WithoutMask': 1}

In [ ]:
def detect_face_mask(img):
  img = image.img_to_array(img)
  img = np.expand_dims(img, axis=0)
  y_pred = model.predict(img)
  y_pred = 1 if y_pred > 0.5 else 0
  return y_pred

In [ ]:
img = image.load_img('mask4.jpg', target_size=(224, 224))
if img is None:
    print("Error: Image not loaded. Check file path or if the image exists.")
else:
    y_pred = detect_face_mask(img)
    print(y_pred)

Error: Image not loaded. Check file path or if the image exists.


In [ ]:
img = image.load_img('mask2.jpg', target_size=(224, 224))
if img is None:
    print("Error: Image not loaded. Check file path or if the image exists.")
else:
    y_pred = detect_face_mask(img)
    print(y_pred)

In [ ]:
img = image.load_img('mask3.jpg', target_size=(224, 224))
if img is None:
    print("Error: Image not loaded. Check file path or if the image exists.")
else:
    y_pred = detect_face_mask(img)
    print(y_pred)

In [ ]:
img = image.load_img('without_mask1.jpg', target_size=(224, 224))
if img is None:
    print("Error: Image not loaded. Check file path or if the image exists.")
else:
    y_pred = detect_face_mask(img)
    print(y_pred)

In [ ]:
img = image.load_img('without_mask2.jpg', target_size=(224, 224))
if img is None:
    print("Error: Image not loaded. Check file path or if the image exists.")
else:
    y_pred = detect_face_mask(img)
    print(y_pred)

In [ ]:
img = image.load_img('without_mask3.jpg', target_size=(224, 224))
if img is None:
    print("Error: Image not loaded. Check file path or if the image exists.")
else:
    y_pred = detect_face_mask(img)
    print(y_pred)

In [39]:
model.save('face_mask.keras')
print("Model saved successfully as face_mask_model.keras")

Model saved successfully as face_mask_model.keras
